***Preparação***

In [1]:
import numpy as np
import time

def generate_data(num_samples=500):
    """Gera dados com a regra condicional de temperatura."""
    X, y = [], []
    for _ in range(num_samples):
        temp, pressure, vibration = np.random.rand(), np.random.rand(), np.random.rand()
        label = 0 # Reprovado por padrão
        
        # A regra de exceção: temperatura na zona de perigo -> reprovação imediata
        if 0.7 <= temp <= 0.9:
            label = 0
        # A regra normal: só checada se a primeira for falsa
        elif pressure > 0.6 and vibration < 0.4:
            label = 1
            
        X.append([temp, pressure, vibration])
        y.append([label])
    return np.array(X), np.array(y)

X_train, y_train = generate_data()

In [2]:
def generate_initial_weights(input_size=3, h1_size=5, h2_size=4, output_size=1, seed=42):
    """Gera pesos/vieses iniciais determinísticos para comparar modelos."""
    rng = np.random.default_rng(seed)
    # Camada H1
    weights_h1 = rng.normal(0, 0.1, size=(input_size, h1_size))
    bias_h1 = np.zeros(h1_size)
    # Camada H2 (densa)
    weights_h2 = rng.normal(0, 0.1, size=(h1_size, h2_size))
    bias_h2 = np.zeros(h2_size)
    # Saída
    weights_out = rng.normal(0, 0.1, size=(h2_size, output_size))
    bias_out = np.zeros(output_size)
    return {
        'weights_h1': weights_h1,
        'bias_h1': bias_h1,
        'weights_h2': weights_h2,
        'bias_h2': bias_h2,
        'weights_out': weights_out,
        'bias_out': bias_out,
    }

initial_weights = generate_initial_weights()

***Modelo de rede neural multi perceptron tradicional***

In [3]:
class StandardMLP:
    """
    Uma Rede Neural Multilayer Perceptron (MLP) tradicional.
    A arquitetura é 3 (entrada) -> 5 (oculta 1) -> 4 (oculta 2) -> 1 (saída).
    Esta rede não possui nenhuma lógica de comporta customizada.
    """
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Os pesos e vieses são armazenados como matrizes e vetores NumPy.
        # Inicializamos com valores pequenos e aleatórios para quebrar a simetria.
        self.weights_h1 = initial_weights["weights_h1"]
        self.bias_h1 = initial_weights["bias_h1"]
        
        # A camada H2 é uma camada densa padrão, assim como a H1.
        self.weights_h2 = initial_weights["weights_h2"]
        self.bias_h2 = initial_weights["bias_h2"]
        
        self.weights_out = initial_weights["weights_out"]
        self.bias_out = initial_weights["bias_out"]
        
        print("Standard MLP (3-5-4-1) criada.")

    def _sigmoid(self, x):
        """Função de ativação sigmoide. Coloca os valores entre 0 e 1."""
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        """Derivada da sigmoide, necessária para o backpropagation."""
        return x * (1 - x)

    def predict(self, inputs):
        """
        Realiza o 'Forward Pass': calcula a predição da rede para uma dada entrada.
        """
        # Da entrada para a camada oculta 1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        
        # Da camada oculta 1 para a camada oculta 2
        self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
        
        # Da camada oculta 2 para a camada de saída
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        
        return final_output

    def train(self, X, y, epochs=2000, learning_rate=0.1):
        """
        Executa o treinamento da rede usando o algoritmo de backpropagation.
        """
        print(f"Iniciando treinamento por {epochs} épocas...")
        start_time = time.time()
        
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- 1. Forward Pass ---
                # A predição é calculada para obter o erro.
                final_output = self.predict(inputs)

                # --- 2. Backward Pass (Cálculo dos Gradientes) ---
                # O erro é a diferença entre o esperado и o obtido.
                error = expected - final_output
                total_error += np.sum(error**2)

                # Calcula o gradiente (delta) para cada camada, de trás para frente.
                d_output = error * self._sigmoid_derivative(final_output)
                
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                error_h1 = d_h2.dot(self.weights_h2.T)
                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- 3. Atualização dos Pesos e Vieses ---
                # Ajusta os parâmetros da rede na direção que minimiza o erro.
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                self.bias_h2 += d_h2 * learning_rate
                
                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate
            
            if (epoch + 1) % 500 == 0:
                print(f"  Época {epoch + 1}/{epochs}, Erro Total: {total_error:.4f}")

        end_time = time.time()
        print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.\n")


# Instanciar a rede neural tradicional

standard_mlp = StandardMLP(initial_weights=initial_weights)

print("--- Iniciando Treinamento ---")
standard_mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO TRADICIONAL")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = standard_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.4f} -> {resultado_final}")

Standard MLP (3-5-4-1) criada.
--- Iniciando Treinamento ---
Iniciando treinamento por 2000 épocas...
  Época 500/2000, Erro Total: 20.0115
  Época 1000/2000, Erro Total: 12.4743
  Época 1500/2000, Erro Total: 8.9349
  Época 2000/2000, Erro Total: 7.6948
Treinamento concluído em 32.62 segundos.

--- Treinamento Concluído ---

      AVALIAÇÃO DO MODELO TRADICIONAL

--- Caso de Teste: Aprovação Normal ---
Entrada: [0.2 0.8 0.1], Resultado Esperado: 1
  Predição da Standard MLP: 0.9984 -> Aprovado

--- Caso de Teste: Reprovação Normal ---
Entrada: [0.3 0.2 0.2], Resultado Esperado: 0
  Predição da Standard MLP: 0.0000 -> Reprovado

--- Caso de Teste: Reprovação por Temperatura (Crítico) ---
Entrada: [0.8 0.9 0.1], Resultado Esperado: 0
  Predição da Standard MLP: 0.4603 -> Reprovado


***Modelo de rede neural multi perceptron com comporta fraca***


In [4]:
class TrainableGatedPerceptron:
    """Neurônio com comporta suave para ser usado na camada H2."""
    def __init__(self, weights, bias):
        self.weights = np.array(weights, dtype=float)
        self.bias = float(bias)
        
        # Parâmetros fixos da comporta para este exemplo
        self.control_input_index = 0 # Será controlado pelo 1º neurônio da H1
        self.inhibit_range = (0.7, 0.9)
        self.sharpness = 50 # Quão "súbita" é a transição da comporta

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _soft_gate_function(self, control_signal):
        A, B = self.inhibit_range
        bump = self._sigmoid(self.sharpness * (control_signal - A)) * \
               self._sigmoid(-self.sharpness * (control_signal - B))
        return 1 - bump

    def forward(self, inputs):
        self.last_inputs = inputs
        control_signal = inputs[self.control_input_index]
        self.last_gate_value = self._soft_gate_function(control_signal)
        
        z = np.dot(self.weights, inputs) + self.bias
        self.last_activation = self._sigmoid(z)
        
        return float(self.last_activation * self.last_gate_value)

class GatedMLP:
    """Nossa MLP com a camada H2 'gated' e um loop de treinamento simplificado."""
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Armazenamento dos pesos e vieses
        self.weights_h1 = initial_weights["weights_h1"]
        self.bias_h1 = initial_weights["bias_h1"]
        
        # A camada H2 é uma lista de neurônios customizados
        self.hidden_layer_2 = [TrainableGatedPerceptron(weights=initial_weights["weights_h2"][:, i], bias=initial_weights["bias_h2"][i]) for i in range(h2_size)]
        
        self.weights_out = initial_weights["weights_out"]
        self.bias_out = initial_weights["bias_out"]

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        return x * (1 - x)

    def predict(self, inputs):
        """Forward pass para predição."""
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        self.h2_output = np.array([float(neuron.forward(self.h1_output)) for neuron in self.hidden_layer_2], dtype=float)
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        return final_output

    def train(self, X, y, epochs=2000, learning_rate=0.1):
        """Loop de treinamento simplificado."""
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- Forward Pass ---
                final_output = self.predict(inputs)

                # --- Backward Pass (Backpropagation Simplificado) ---
                error = expected - final_output
                total_error += error**2

                # Gradiente da camada de saída
                d_output = error * self._sigmoid_derivative(final_output)

                # Gradiente da camada H2
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                # Gradiente da camada H1
                # Aqui, propagamos o erro através dos pesos de cada neurônio H2
                error_h1 = np.zeros(self.weights_h1.shape[1])
                for i, neuron in enumerate(self.hidden_layer_2):
                    error_h1 += d_h2[i] * neuron.weights * neuron.last_gate_value

                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- Atualização dos Pesos e Vieses ---
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                for i, neuron in enumerate(self.hidden_layer_2):
                    neuron.weights += neuron.last_inputs * d_h2[i] * learning_rate
                    neuron.bias += d_h2[i] * learning_rate

                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate

            if (epoch + 1) % 500 == 0:
                print(f"Época {epoch + 1}/{epochs}, Erro: {total_error[0]:.4f}")


mlp = GatedMLP(initial_weights=initial_weights)

# Treinar a rede
print("--- Iniciando Treinamento ---")
mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO GATE")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.4f} -> {resultado_final}")

--- Iniciando Treinamento ---
Época 500/2000, Erro: 7.1653
Época 1000/2000, Erro: 6.1829
Época 1500/2000, Erro: 5.2462
Época 2000/2000, Erro: 5.1255
--- Treinamento Concluído ---

      AVALIAÇÃO DO MODELO GATE

--- Caso de Teste: Aprovação Normal ---
Entrada: [0.2 0.8 0.1], Resultado Esperado: 1
  Predição da Standard MLP: 0.9998 -> Aprovado

--- Caso de Teste: Reprovação Normal ---
Entrada: [0.3 0.2 0.2], Resultado Esperado: 0
  Predição da Standard MLP: 0.0000 -> Reprovado

--- Caso de Teste: Reprovação por Temperatura (Crítico) ---
Entrada: [0.8 0.9 0.1], Resultado Esperado: 0
  Predição da Standard MLP: 0.0000 -> Reprovado


***Modelo de rede neural multi perceptron com comporta forte***

In [ ]:
class SmartMLP:
    """
    MLP com opção de ignorar parcial ou totalmente a camada oculta H2.
    Quando use_h2=False, a saída conecta diretamente H1 -> Saída por pesos de bypass.
    Quando use_h2=True, é possível ativar um subconjunto dos neurônios de H2 via máscara (h2_mask).
    Durante o treinamento (try_both=True), testamos AUTOMATICAMENTE:
      - bypass (0 neurônios H2)
      - TODAS as combinações de 1..h2_size neurônios ativos em H2,
    mantendo a configuração (máscara) que produzir o menor erro total.
    """
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1, use_h2=True):
        # Pesos/vieses das camadas tradicionais (fornecidos externamente)
        self.weights_h1 = initial_weights["weights_h1"]
        self.bias_h1 = initial_weights["bias_h1"]
        self.weights_h2 = initial_weights["weights_h2"]
        self.bias_h2 = initial_weights["bias_h2"]
        self.weights_out = initial_weights["weights_out"]
        self.bias_out = initial_weights["bias_out"]

        # Pesos/vieses para o caminho de bypass (H1 -> Saída). Inicializamos com zeros para
        # garantir determinismo. Eles serão ajustados se use_h2=False vencer no treinamento.
        import numpy as _np
        self.weights_bypass_out = _np.zeros((h1_size, output_size))
        self.bias_bypass_out = _np.zeros(output_size)

        # Máscara de H2 (1.0 = ativo, 0.0 = inativo). Por padrão, todos ativos.
        self.h2_mask = _np.ones(h2_size)

        self.use_h2 = bool(use_h2)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        return x * (1 - x)

    def predict(self, inputs, use_h2=None):
        """
        Forward pass que respeita a flag de uso da H2 e uma possível máscara de neurônios.
        - Se use_h2=True: Entrada -> H1(sigmoid) -> H2(sigmoid & máscara) -> Out(sigmoid)
        - Se use_h2=False: Entrada -> H1(sigmoid) -> Out_bypass(sigmoid)
        """
        if use_h2 is None:
            use_h2 = self.use_h2

        # Camada H1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)

        if use_h2:
            # Camada H2
            self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
            # Aplicar máscara de H2 (neurônios inativos não contribuem)
            self.h2_used = self.h2_output * self.h2_mask
            # Saída final via H2 mascarada
            final_output = self._sigmoid(np.dot(self.h2_used, self.weights_out) + self.bias_out)
        else:
            # Ignoramos H2 e usamos o caminho de bypass H1 -> Saída
            final_output = self._sigmoid(np.dot(self.h1_output, self.weights_bypass_out) + self.bias_bypass_out)

        return final_output

    def _snapshot(self):
        """Cria um snapshot dos pesos atuais para comparar cenários."""
        return {
            'weights_h1': self.weights_h1.copy(),
            'bias_h1': self.bias_h1.copy(),
            'weights_h2': self.weights_h2.copy(),
            'bias_h2': self.bias_h2.copy(),
            'weights_out': self.weights_out.copy(),
            'bias_out': self.bias_out.copy(),
            'weights_bypass_out': self.weights_bypass_out.copy(),
            'bias_bypass_out': self.bias_bypass_out.copy(),
            'h2_mask': self.h2_mask.copy(),
            'use_h2': bool(self.use_h2),
        }

    def _restore(self, snap):
        """Restaura pesos a partir de um snapshot."""
        self.weights_h1 = snap['weights_h1'].copy()
        self.bias_h1 = snap['bias_h1'].copy()
        self.weights_h2 = snap['weights_h2'].copy()
        self.bias_h2 = snap['bias_h2'].copy()
        self.weights_out = snap['weights_out'].copy()
        self.bias_out = snap['bias_out'].copy()
        self.weights_bypass_out = snap['weights_bypass_out'].copy()
        self.bias_bypass_out = snap['bias_bypass_out'].copy()
        self.h2_mask = snap['h2_mask'].copy()
        self.use_h2 = bool(snap['use_h2'])

    def _train_single(self, X, y, epochs, learning_rate, use_h2_flag, h2_mask=None):
        """
        Treina um único cenário e retorna o erro total ao final.
        - use_h2_flag=True: utiliza H2 com máscara (se fornecida).
        - use_h2_flag=False: utiliza caminho de bypass (H1->Saída).
        """
        import numpy as _np
        self.use_h2 = use_h2_flag
        if use_h2_flag:
            if h2_mask is None:
                # Se não foi passada máscara, ativa todos os neurônios de H2
                self.h2_mask = _np.ones(self.weights_h2.shape[1])
            else:
                self.h2_mask = _np.array(h2_mask, dtype=float)
        total_error = 0.0
        for epoch in range(epochs):
            epoch_error = 0.0
            for inputs, expected in zip(X, y):
                final_output = self.predict(inputs, use_h2=use_h2_flag)
                error = expected - final_output
                epoch_error += float(np.sum(error ** 2))

                d_output = error * self._sigmoid_derivative(final_output)

                if use_h2_flag:
                    # Backprop com H2 (respeitando a máscara)
                    error_h2 = d_output.dot(self.weights_out.T)
                    d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                    # Zerar gradientes dos neurônios inativos de H2
                    d_h2 = d_h2 * self.h2_mask

                    error_h1 = d_h2.dot(self.weights_h2.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                    # Atualizações (usar h2_used para respeitar máscara no gradiente de saída)
                    self.weights_out += self.h2_used.reshape(-1, 1) * d_output * learning_rate
                    self.bias_out += d_output * learning_rate

                    self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                    self.bias_h2 += d_h2 * learning_rate

                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
                else:
                    # Backprop sem H2 (via caminho de bypass)
                    error_h1 = d_output.dot(self.weights_bypass_out.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                    # Atualizações para bypass e H1
                    self.weights_bypass_out += self.h1_output.reshape(-1, 1) * d_output * learning_rate
                    self.bias_bypass_out += d_output * learning_rate

                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate

            # Log de treinamento a cada 500 épocas e na última época
            if ((epoch + 1) % 500 == 0) or ((epoch + 1) == epochs):
                if use_h2_flag:
                    mask_list = self.h2_mask.astype(int).tolist()
                    false_idx = [i for i, v in enumerate(mask_list) if v == 0]
                else:
                    false_idx = list(range(self.weights_h2.shape[1]))
                print(f"Época {epoch+1}/{epochs}, use_h2={use_h2_flag}, h2_false={false_idx}, Erro: {epoch_error:.4f}")

            total_error = epoch_error
        return total_error

    def train(self, X, y, epochs=2000, learning_rate=0.1, try_both=True):
        """
        Executa o treinamento.
        - Se try_both=True: testa bypass (0 H2) e TODAS as combinações possíveis de neurônios de H2 (1..h2_size),
          escolhe o menor erro final e mantém os pesos e a máscara referentes ao melhor cenário.
        - Se try_both=False: treina apenas no modo atual self.use_h2 e máscara atual self.h2_mask.
        """
        import time as _time
        import itertools as _itertools
        import numpy as _np
        start = _time.time()

        if not try_both:
            final_error = self._train_single(X, y, epochs, learning_rate, self.use_h2, h2_mask=self.h2_mask)
            print(f"Treinamento concluído (use_h2={self.use_h2}, mask={self.h2_mask.astype(int).tolist()}) com erro final: {final_error:.4f}")
            print(f"Tempo: {_time.time() - start:.2f}s")
            return {'use_h2': self.use_h2, 'final_error': final_error, 'h2_mask': self.h2_mask.copy()}

        # Snapshot inicial para replicar condições em todos os cenários
        base = self._snapshot()

        results = []

        # Cenário 0: bypass (use_h2=False)
        self._restore(base)
        err_bypass = self._train_single(X, y, epochs, learning_rate, use_h2_flag=False)
        snap_bypass = self._snapshot()
        results.append(('bypass', err_bypass, None, snap_bypass))

        # Demais cenários: todas as combinações de 1..h2_size neurônios ativos em H2
        h2_size = self.weights_h2.shape[1]
        indices = list(range(h2_size))
        for k in range(h2_size, 0, -1):
            for combo in _itertools.combinations(indices, k):
                mask = _np.zeros(h2_size, dtype=float)
                mask[list(combo)] = 1.0
                self._restore(base)
                err = self._train_single(X, y, epochs, learning_rate, use_h2_flag=True, h2_mask=mask)
                snap = self._snapshot()
                results.append((f'h2_k={k}', err, mask.copy(), snap))

        # Escolha do melhor cenário
        best = min(results, key=lambda t: t[1])
        label, chosen_err, chosen_mask, chosen_snap = best

        self._restore(chosen_snap)
        if label == 'bypass':
            self.use_h2 = False
            self.h2_mask = _np.zeros(h2_size)
        else:
            self.use_h2 = True
            self.h2_mask = chosen_mask.copy()

        print("Resultados SmartMLP:")
        print(f"  bypass (use_h2=False) -> erro final: {err_bypass:.4f}")
        # Opcional: imprimir um resumo de algumas combinações testadas
        # Aqui, mostramos apenas as melhores 3 combinações com H2 para evitar poluição de logs
        only_h2 = [(m, e) for (lab, e, m, s) in results if lab != 'bypass']
        only_h2_sorted = sorted(only_h2, key=lambda t: t[1])
        for i, (m, e) in enumerate(only_h2_sorted[:3]):
            print(f"  h2_mask_top{i+1}={m.astype(int).tolist()} -> erro final: {e:.4f}")
        print(f"  Escolhido: use_h2={self.use_h2}, mask={self.h2_mask.astype(int).tolist()} (erro {chosen_err:.4f})")
        print(f"Tempo total: {_time.time() - start:.2f}s")
        return {
            'use_h2': self.use_h2,
            'final_error': chosen_err,
            'h2_mask': self.h2_mask.copy(),
            'err_bypass': err_bypass,
        }


In [17]:
smart_mlp = SmartMLP(initial_weights=initial_weights)
# Treinar a rede
print("--- Iniciando Treinamento ---")
smart_mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO GATE")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = smart_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.4f} -> {resultado_final}")

--- Iniciando Treinamento ---
Época 500/500, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 7.0827
Época 500/500, use_h2=True, h2_false=[], Erro: 5.5362
Época 500/500, use_h2=True, h2_false=[3], Erro: 6.4234
Época 500/500, use_h2=True, h2_false=[2], Erro: 5.3077
Época 500/500, use_h2=True, h2_false=[1], Erro: 5.2805
Época 500/500, use_h2=True, h2_false=[0], Erro: 5.1338
Época 500/500, use_h2=True, h2_false=[2, 3], Erro: 5.8901
Época 500/500, use_h2=True, h2_false=[1, 3], Erro: 5.9279
Época 500/500, use_h2=True, h2_false=[1, 2], Erro: 5.0067
Época 500/500, use_h2=True, h2_false=[0, 3], Erro: 6.6089
Época 500/500, use_h2=True, h2_false=[0, 2], Erro: 4.8898
Época 500/500, use_h2=True, h2_false=[0, 1], Erro: 4.8880
Época 500/500, use_h2=True, h2_false=[1, 2, 3], Erro: 6.6078
Época 500/500, use_h2=True, h2_false=[0, 2, 3], Erro: 6.6687
Época 500/500, use_h2=True, h2_false=[0, 1, 3], Erro: 6.6702
Época 500/500, use_h2=True, h2_false=[0, 1, 2], Erro: 5.6393
Resultados SmartMLP:
  bypass (use_h2=F